# M2 远程 PPO 云 Worker — Colab 冒烟测试

连接到 hub 的 cloudflared tunnel，拉取 job 执行 PPO 更新。

## 使用前

1. 确保 hub 端已启动：`hub-server` + `cloudflared tunnel` + `training loop`
2. 在下方的 `⚙️ 参数配置` 单元格填入当前 tunnel URL 和 token
3. 依次运行各单元格


---
## ⚙️ 参数配置


In [ ]:
# @title 填入 hub 连接信息
import time

# 手动填入以下参数
# HUB_URL 和 HUB_TOKEN 由 hub 端提供，见 hub 的 rl-config.json 中 remote_hub_url / remote_token
HUB_URL = "https://tied-salt-tried-proceedings.trycloudflare.com"
HUB_TOKEN = "YOUR_TOKEN_HERE"

# 保活配置
KEEPALIVE_HOURS = 2
POLL_INTERVAL_SEC = 5
# 设备类型：自动检测（torch.cuda.is_available()）

print(f"[{time.strftime('%H:%M:%S')}] HUB_URL = '{HUB_URL}'")
print(f"[{time.strftime('%H:%M:%S')}] HUB_TOKEN len = {len(HUB_TOKEN)}")
print(f"[{time.strftime('%H:%M:%S')}] Keepalive = {KEEPALIVE_HOURS}h")


---
## 1. 安装依赖


In [ ]:
import subprocess
import sys

import torch


def run(cmd, **kw):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, **kw)

run(f"{sys.executable} -m pip install --quiet torch numpy")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[{time.strftime('%H:%M:%S')}] torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[{time.strftime('%H:%M:%S')}]   device: {torch.cuda.get_device_name(0)}")

print(f"[{time.strftime('%H:%M:%S')}] Dependencies ready")


---
## 2. 保活线程（防止 Colab 90min 超时）

Colab 免费版约 90min 回收空闲会话。通过定时 ping 界面保持活跃。


In [ ]:
import threading

from IPython.display import Javascript
from IPython.display import display as ipy_display

KEEPALIVE_STOP = threading.Event()

def keepalive_loop():
    """每 60s 点一次 Colab 的 connect 防止超时。"""
    n = 0
    while not KEEPALIVE_STOP.is_set():
        try:
            ipy_display(Javascript("""
                function clickConnect() {
                    document.querySelector("colab-connect-button")?.click();
                }
                setTimeout(clickConnect, 1000);
            """))
            n += 1
        except Exception:
            pass
        KEEPALIVE_STOP.wait(60)
    print(f"[{time.strftime('%H:%M:%S')}] Keepalive stopped after {n} pings")

th = threading.Thread(target=keepalive_loop, daemon=True, name="keepalive")
th.start()
print(f"[{time.strftime('%H:%M:%S')}] Keepalive thread started (every 60s, max {KEEPALIVE_HOURS}h)")


---
## 3. 下载 code.zip 并启动远程 PPO Worker

从 hub 下载 code.zip（hub 启动时打包的代码快照）→ 解压到 `sys.path` → 启动 worker 轮询。

不需要 git clone，代码一致性由 `code_sha256` 保证。

> 可以随时中断此单元格（`Runtime → Interrupt execution`），worker 会优雅退出。


In [ ]:
import io
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

WORK_DIR = Path("/content/remote-worker")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"[{time.strftime('%H:%M:%S')}] [colab] Downloading code.zip from {HUB_URL}/code...")
req = urllib.request.Request(
    f"{HUB_URL.rstrip('/')}/code",
    headers={"Authorization": f"Bearer {HUB_TOKEN}"},
)
try:
    with urllib.request.urlopen(req, timeout=120) as resp:
        code_raw = resp.read()
        print(f"[{time.strftime('%H:%M:%S')}] [colab] code.zip: {len(code_raw)} bytes (HTTP {resp.status})")
except urllib.error.HTTPError as e:
    print(f"[{time.strftime('%H:%M:%S')}] [colab] FAILED to download code.zip: HTTP {e.code} — {e.read().decode()[:200]}")
    raise SystemExit(1) from None
except Exception as e:
    print(f"[{time.strftime('%H:%M:%S')}] [colab] FAILED to connect to hub: {e}")
    raise SystemExit(1) from None

# 解压到 sys.path，后续 import remote.worker 从 code.zip 加载
CODE_DIR = Path("/content/worker-code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(code_raw)) as z:
    z.extractall(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
print(f"[{time.strftime('%H:%M:%S')}] [colab] code.zip extracted -> {CODE_DIR} (sys.path[0])")

from remote.worker import worker_loop

print(f"[{time.strftime('%H:%M:%S')}] [colab] Testing connectivity to hub...")
req = urllib.request.Request(
    f"{HUB_URL.rstrip('/')}/ping",
    headers={"Authorization": f"Bearer {HUB_TOKEN}"},
)
try:
    with urllib.request.urlopen(req, timeout=15) as resp:
        print(f"[{time.strftime('%H:%M:%S')}] [colab] hub HTTP {resp.status}")
except Exception as e:
    print(f"[{time.strftime('%H:%M:%S')}] [colab] CONNECTION FAILED: {e}")
    print(f"[{time.strftime('%H:%M:%S')}] [colab] Check HUB_URL and try again")
    raise SystemExit(1) from None

print(f"[{time.strftime('%H:%M:%S')}] [colab] Starting worker loop...")
t_start = time.time()

try:
    n = worker_loop(
        HUB_URL,
        HUB_TOKEN,
        work_dir=WORK_DIR,
        device=DEVICE,
        torch_threads=0,
        poll_sec=POLL_INTERVAL_SEC,
        once=False,
        max_idle_sec=KEEPALIVE_HOURS * 3600,
    )
    print(f"\n[{time.strftime('%H:%M:%S')}] [colab] Worker exited: {n} job(s) processed")
except KeyboardInterrupt:
    print(f"\n[{time.strftime('%H:%M:%S')}] [colab] Worker interrupted by user")
finally:
    KEEPALIVE_STOP.set()

elapsed = time.time() - t_start
print(f"[{time.strftime('%H:%M:%S')}] [colab] Session duration: {elapsed/60:.1f} min")


---
## 4. 停止保活

如果提前中断了 worker，运行此单元格停止保活线程。


In [ ]:
KEEPALIVE_STOP.set()
print(f"[{time.strftime('%H:%M:%S')}] Keepalive stopped")


---
## 附录：预期日志

hub 端：
```
[hub-server] "GET /jobs/next HTTP/1.1" 200 -
[hub-server] "GET /jobs/{id}/payload HTTP/1.1" 200 -
[hub-server] "POST /jobs/{id}/result HTTP/1.1" 200 -
```

worker 端：
```
[worker] job {id} claimed — downloading payload
[worker] job {id}: code.zip unpacked (N bytes, M .py files) -> sys.path[0]
[worker] job {id}: PPO done in {sec}s, steps={n} chunks={m} kl={k}
[worker] job {id} done — result accepted
```
